### Save MRMS data for different variables and date ranges

MRMS variable names and descriptions: https://www.nssl.noaa.gov/projects/mrms/operational/tables.php

In [1]:
import s3fs
import pygrib
import gzip
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

In [2]:
def load_mrms_file(datetime, variable, grib_table, region="CONUS"):
    ts = pd.Timestamp(datetime)
    ts_date_str = ts.strftime("%Y%m%d")
    ts_datetime_str = ts.strftime("%Y%m%d-%H%M%S")
    s3 = s3fs.S3FileSystem(anon=True)
    filename = f"s3://noaa-mrms-pds/{region}/{variable}/{ts_date_str}/MRMS_{variable}_{ts_datetime_str}.grib2.gz"
    with s3.open(filename, "rb") as f:
        f_data = f.read()
    data = pygrib.fromstring(gzip.decompress(f_data))
    grib_attrs = (
        grib_table.loc[
            (grib_table["Category"] == data["parameterCategory"])
            & (grib_table["Parameter"] == data["parameterNumber"])
        ]
        .T.iloc[:, 0]
        .dropna()
        .to_dict()
    )
    xr_data = xr.DataArray(
        data=np.expand_dims(data.values[::-1], axis=0),
        coords={
            "time": [datetime],
            "lat": data.latitudes[:: data.Nx][::-1],
            "lon": data.longitudes[: data.Nx],
        },
        name=variable,
        attrs=grib_attrs,
    )
    return xr_data

In [3]:
def save_mrms_data(s_date_str, e_date_str, variable, grib_table_url):
    grib_table = pd.read_csv(grib_table_url)
    dates = pd.date_range(s_date_str, e_date_str, freq="1d")

    time_steps = []
    for date in tqdm(dates):
        print(date)
        try:
            mrms_full_step = load_mrms_file(date, variable, grib_table)
        except Exception as e:
            print(e)
            print(date, variable)
        time_steps.append(
            mrms_full_step.sel(
                lon=slice(360 - 124.7, 360 - 113.9, None), lat=slice(32.3, 42.1, None)
            )
        )
    combined = xr.concat(time_steps, dim="time")
    combined.to_netcdf(
        f"/glade/campaign/cisl/aiml/cat_flood/mrms/{variable}_{s_date_str.replace(' ', 'T')}_{e_date_str.replace(' ', 'T')}.nc"
    )

In [ ]:
s_date_str = "2024-03-15 00:00"
e_date_str = "2024-04-29 00:00"
# variable = "MultiSensor_QPE_01H_Pass2_00.00"
variable = "FLASH_HP_MAXUNITSTREAMFLOW_00.00"
grib_table_url = "https://raw.githubusercontent.com/NOAA-National-Severe-Storms-Laboratory/mrms-support/refs/heads/main/GRIB2_TABLES/UserTable_MRMS_v12.2.csv"

save_mrms_data(s_date_str, e_date_str, variable, grib_table_url)

  0%|          | 0/46 [00:00<?, ?it/s]

2024-03-15 00:00:00


  2%|▏         | 1/46 [00:02<01:33,  2.07s/it]

2024-03-16 00:00:00


  4%|▍         | 2/46 [00:02<00:57,  1.31s/it]

2024-03-17 00:00:00


... (44 outputs left out)

 54%|█████▍    | 25/46 [00:21<00:16,  1.25it/s]

2024-04-09 00:00:00


 57%|█████▋    | 26/46 [00:22<00:16,  1.25it/s]

2024-04-10 00:00:00


### Begin home location to MRMS matching

#### Load home location file

### Load MRMS variable files

### Match home locations to nearest MRMS data

In [1]:
import yaml
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
with open("../config/zips_counties.yaml", 'r') as stream:
    out = yaml.load(stream, Loader=yaml.FullLoader)
    zips = [str(z) for z in out["zips_LA"] + out["zips_SB"] + out["zips_VE"]]

In [3]:
# variable = "MultiSensor_QPE_01H_Pass2_00.00"
variable = "FLASH_HP_MAXUNITSTREAMFLOW_00.00"
s_date_str = "2024-03-29T12:00"
e_date_str = "2024-04-01T12:00"
run_date = "20250728"
mrms_fn = f"/glade/campaign/cisl/aiml/cat_flood/mrms_data/2024/veraset_users_mrms_{variable}_{s_date_str}_{e_date_str}_{run_date}.parquet"
mrms_df = pd.read_parquet(mrms_fn)
mrms_df = mrms_df[mrms_df.zipcode.isin(zips)]
print(mrms_df.shape)

(5024, 13)


In [9]:
variable = "MultiSensor_QPE_01H_Pass2_00.00"
# variable = "FLASH_HP_MAXUNITSTREAMFLOW_00.00"
s_date_str = "2024-03-2T00:00"
e_date_str = "2024-04-01T00:00"
run_date = "20250728"
mrms_fn = f"/glade/campaign/cisl/aiml/cat_flood/mrms_data/2024/veraset_users_mrms_{variable}_{s_date_str}_{e_date_str}_{run_date}.parquet"
mrms_df = pd.read_parquet(mrms_fn)
mrms_df = mrms_df[mrms_df.zipcode.isin(zips)]
print(mrms_df.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/glade/campaign/cisl/aiml/cat_flood/mrms_data/2024/veraset_users_mrms_MultiSensor_QPE_01H_Pass2_00.00_2024-03-27T00:00_2024-04-02T00:00_20250729.parquet'

In [4]:
data_dir = "/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20250720/"
fns = [f"veraset_2024-03-{date:02}_PST_chunked.parquet" for date in range(15,19)]
fns += [f"veraset_2024-03-{date:02}_PST_chunked.parquet" for date in range(29,32)]
fns += [f"veraset_2024-04-{date:02}_PST_chunked.parquet" for date in range(1,2)]
AR_event_dates = [f"2024-03-{date:02}" for date in range(29,32)] + [f"2024-04-{date:02}" for date in range(1,2)]
metrics_df = pd.concat(
    pd.read_parquet(data_dir+pf)
    for pf in fns)
metrics_df = metrics_df[metrics_df.zipcode.isin(zips)]
metrics_df["event"] = np.where(metrics_df.date.isin(AR_event_dates), "AR", "base")
print(metrics_df.shape)

(34679, 17)


/glade/derecho/scratch/ggantos/tmp/ipykernel_61994/722451371.py:10: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  metrics_df["event"] = np.where(metrics_df.date.isin(AR_event_dates), "AR", "base")


In [4]:
data_dir = "/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20250720/"
fns = [f"veraset_2024-03-{date:02}_PST_chunked.parquet" for date in range(15,19)]
fns += [f"veraset_2024-03-{date:02}_PST_chunked.parquet" for date in range(29,32)]
fns += [f"veraset_2024-04-{date:02}_PST_chunked.parquet" for date in range(1,2)]
AR_event_dates = [f"2024-03-{date:02}" for date in range(29,32)] + [f"2024-04-{date:02}" for date in range(1,2)]
metrics_df = pd.concat(
    pd.read_parquet(data_dir+pf)
    for pf in fns)
metrics_df = metrics_df[metrics_df.zipcode.isin(zips)]
metrics_df["event"] = np.where(metrics_df.date.isin(AR_event_dates), "AR", "base")
print(metrics_df.shape)

(34679, 17)


/glade/derecho/scratch/ggantos/tmp/ipykernel_62283/722451371.py:10: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  metrics_df["event"] = np.where(metrics_df.date.isin(AR_event_dates), "AR", "base")


In [5]:
matching_uids = np.intersect1d(metrics_df.uid, mrms_df.index)
matching_uids.shape

(4483,)

In [5]:
matching_uids = np.intersect1d(metrics_df.uid, mrms_df.index)
matching_uids.shape

(4483,)

In [6]:
def get_precip_category(uid, mrms_df, variable):
    high_uids = mrms_df[mrms_df[variable] == 'high'].index
    if uid in high_uids:
        return 'high'
    else:
        return 'low'

In [7]:
# people with no precip vs people who experienced more than 0mm precip and work way up to 50-50
plt.rcParams.update({'font.size': 14})

mrms_metrics = ["mrms_sum", "mrms_max", "mrms_mean"][0:1]
if variable == "MultiSensor_QPE_01H_Pass2_00.00":
    mrms_metric_label = "precip"
    mrms_labels = [
        "Cumulative Precipitation (mm)",
        "Max Precipitation (mm)",
        "Mean Precipitation (mm)",
    ][0:1]
elif variable == "FLASH_HP_MAXUNITSTREAMFLOW_00.00":
    mrms_metric_label = "flood"
    mrms_labels = [
        "Cumulative Flood (m3/s/km2)",
        "Max Flood (m3/s/km2)",
        "Mean Flood (m3/s/km2)",
    ][0:1]
mobility_metrics = [
    "number_of_visits",
    "distance_straight_line",
    "radius_of_gyration",
    "num_stops",
    "max_distance_from_home",
]
mobility_labels = [
    "Number of Visits ",
    "Cumulative Distance",
    "Radius of Gyration",
    "Number of Stops",
    "Maxmimum Distance from Home",
]
percentiles = np.arange(0.1, 1, 0.1)
percentile_keys = [f"{int(p*100)}%" for p in percentiles][-1:]

for i, mrms_metric in enumerate(mrms_metrics):
    metric_describe = mrms_df[mrms_metric].describe(percentiles=percentiles)
    mrms_metric_cat = f"{mrms_metric}_category"
    for perc in percentile_keys:
        mrms_df[mrms_metric_cat] = pd.cut(
            mrms_df[mrms_metric],
            bins=[0, metric_describe[perc], float("Inf")],
            labels=["low", "high"],
        )
        metrics_df[mrms_metric_cat] = metrics_df.uid.apply(
            get_precip_category, mrms_df=mrms_df, variable=mrms_metric_cat
        )
        users_high = mrms_df[mrms_df[mrms_metric_cat] == "high"].shape[0]
        users_low = mrms_df[mrms_df[mrms_metric_cat] == "low"].shape[0]
        for j, mobility_metric in enumerate(mobility_metrics):
            means = metrics_df.groupby([mrms_metric_cat, "event"])[
                mobility_metric
            ].mean()
            fig = plt.figure(figsize=(10, 8))
            gs = fig.add_gridspec(nrows=2, ncols=2)
            ax1 = fig.add_subplot(gs[0, :])
            ax2 = fig.add_subplot(gs[1, 0])
            ax3 = fig.add_subplot(gs[1, 1], sharey=ax2)
            fig.suptitle(
                f"{metrics_df['date'].dt.strftime('%Y').unique()[0]} - {perc} low - {100-int(perc[:2])}% high",
                # f"{metrics_df['date'].dt.strftime('%Y').unique()[0]} - {perc} low ({users_low:,} users) - {100-int(perc[:2])}% high ({users_high:,} users)",
                fontsize=16,
                fontweight='bold',
            )
            fig.set_constrained_layout(True)
            x_dates = metrics_df["date"].dt.strftime("%m-%d").sort_values().unique()
            flierprops = dict(marker=".", alpha=0.75, markersize=4)

            sns.boxplot(
                data=mrms_df,
                x=f"{mrms_metric_cat}",
                y=f"{mrms_metric}",
                hue=f"{mrms_metric_cat}",
                flierprops=flierprops,
                orient="v",
                ax=ax1,
                legend=False,
            ).set(xlabel="")
            ax1.set_ylabel(f"{mrms_labels[i]}", fontdict={'weight': 'bold'})
            sns.boxplot(
                y=metrics_df[metrics_df[mrms_metric_cat] == "low"][mobility_metric],
                x=metrics_df[metrics_df[mrms_metric_cat] == "low"]["date"],
                color=sns.color_palette()[0],
                flierprops=flierprops,
                orient="v",
                ax=ax2,
            ).set(xlabel="")
            ax2.set_ylabel(f"{mobility_labels[j]}", fontdict={'weight': 'bold'})
            ax2.axhline(
                y=means["low", "base"],
                xmin=0.0,
                xmax=0.5,
                color=sns.color_palette()[2],
                ls="--",
                label=f"base: {means['low', 'base']:.0f}",
            )
            ax2.axhline(
                y=means["low", "AR"],
                xmin=0.5,
                xmax=1.0,
                color=sns.color_palette()[3],
                ls="--",
                label=f"AR: {means['low', 'AR']:.0f}",
            )
            ax2.set_yscale("log")
            ax2.xaxis.set_ticks(ax2.get_xticks())
            ax2.set_xticklabels(x_dates, rotation=45, ha="right")
            ax2.legend(bbox_to_anchor=(0.5, 1.12), loc="upper center", ncol=2)
            sns.boxplot(
                y=metrics_df[metrics_df[mrms_metric_cat] == "high"][mobility_metric],
                x=metrics_df[metrics_df[mrms_metric_cat] == "high"]["date"],
                color=sns.color_palette()[1],
                flierprops=flierprops,
                orient="v",
                ax=ax3,
            ).set(xlabel="", ylabel="")
            ax3.axhline(
                y=means["high", "base"],
                xmin=0.0,
                xmax=0.5,
                color=sns.color_palette()[2],
                ls="--",
                label=f"base: {means['high', 'base']:.0f}",
            )
            ax3.axhline(
                y=means["high", "AR"],
                xmin=0.5,
                xmax=1.0,
                color=sns.color_palette()[3],
                ls="--",
                label=f"AR: {means['high', 'AR']:.0f}",
            )
            ax3.set_yscale("log")
            ax3.xaxis.set_ticks(ax3.get_xticks())
            ax3.set_xticklabels(x_dates, rotation=45, ha="right")
            ax3.legend(bbox_to_anchor=(0.5, 1.12), loc="upper center", ncol=2)

            fig.savefig(
                f"mrms/plot_{mrms_metric}_{mrms_metric_label}_{mobility_metric}_{perc}_logy.png",
                transparent=True,
            )
            fig.show()
            plt.close(fig)